# Milestone 3 lab, notebook 1 of 8: One order, one row

## Where we are

| Notebook | What you do |
|---|---|
| **1. One order, one row** (you are here) | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

**Before you start,** open [`milestone_3_flow.html`](../milestone_3_flow.html) in a browser.

Milestone 2 ended with a report that counts changed orders twice. The raw landing table was right to keep both versions. What is missing is a table above it with **one row per order**. This notebook builds it, three ways, on the same rows.

In [1]:
from lab_helpers import lab_warehouse, land_like_milestone_2, money, show_doc, table, true_day_totals

con = lab_warehouse(fresh=True)
land_like_milestone_2(con, "2026-06-08", "2026-06-09")

Landed 226 rows for the morning of 2026-06-08.
Landed  29 rows for the morning of 2026-06-09.


That is the state at the end of milestone 2: two mornings, landed raw. Count the rows, and count the orders.

In [2]:
table(con, """
    SELECT count(*) AS rows_landed,
           count(DISTINCT payload->>'order_id') AS different_orders
    FROM raw.orders_landing
""")

,rows_landed,different_orders
0,255,251


255 rows for 251 orders. Four orders are there twice. To keep the SQL short, make a view that shows the landed rows as columns.

In [3]:
con.execute("""
    CREATE VIEW landed AS
    SELECT landing_id, run_id,
           (payload->>'order_id')::BIGINT                                            AS order_id,
           payload->>'status'                                                        AS status,
           ((payload->>'placed_at')::TIMESTAMPTZ AT TIME ZONE 'America/Chicago')::DATE AS order_date,
           (payload->>'total_cents')::BIGINT                                         AS total_cents,
           (payload->>'refunded_cents')::BIGINT                                      AS refunded_cents,
           (payload->>'updated_at')::TIMESTAMPTZ                                     AS updated_at
    FROM raw.orders_landing
""")
table(con, "SELECT * FROM landed WHERE order_id = 1130 ORDER BY landing_id")

,landing_id,run_id,order_id,status,order_date,total_cents,refunded_cents,updated_at
0,137,2026-06-08_1,1130,delivered,2026-06-05,1475,0,2026-06-05 20:45:00+00:00
1,232,2026-06-09_1,1130,delivered,2026-06-05,1475,578,2026-06-08 16:00:00+00:00


## Three ways to write Tuesday's rows

Monday's 226 orders are in a table. Tuesday's run brings 29 rows. How do you write them? There are three basic ways. Try each one on its own copy.

In [4]:
def fresh_table(name):
    """A new orders table that holds Monday's rows."""
    con.execute(f"DROP TABLE IF EXISTS {name}")
    con.execute(f"""
        CREATE TABLE {name} AS
        SELECT order_id, status, order_date, total_cents, refunded_cents, updated_at
        FROM landed WHERE run_id = '2026-06-08_1'
    """)

TUESDAY = "SELECT order_id, status, order_date, total_cents, refunded_cents, updated_at FROM landed WHERE run_id = '2026-06-09_1'"

def describe(name):
    return table(con, f"""
        SELECT '{name}' AS table_name, count(*) AS rows, count(DISTINCT order_id) AS different_orders,
               count(*) FILTER (WHERE order_id = 1130) AS rows_for_order_1130,
               max(refunded_cents) FILTER (WHERE order_id = 1130) AS refund_on_1130
        FROM {name}
    """)

### Way 1: append

Add the new rows at the end. This is what milestone 2 did.

In [5]:
fresh_table("orders_append")
con.execute(f"INSERT INTO orders_append {TUESDAY}")
describe("orders_append")

,table_name,rows,different_orders,rows_for_order_1130,refund_on_1130
0,orders_append,255,251,2,578


Two rows for order 1130. The refund is there, and so is the old version.

### Way 2: overwrite

Empty the table, then load the new rows. Another name is a full refresh.

In [6]:
fresh_table("orders_overwrite")
with con.transaction():
    con.execute("TRUNCATE orders_overwrite")
    con.execute(f"INSERT INTO orders_overwrite {TUESDAY}")
describe("orders_overwrite")

,table_name,rows,different_orders,rows_for_order_1130,refund_on_1130
0,orders_overwrite,29,29,1,578


One row for order 1130, with the refund. But the table has 29 rows. **222 orders are gone.**

Overwrite is correct only when the new rows are the **whole** table. That was milestone 1: a full load, then replace everything. With an incremental load it throws away every order that did not change.

### Way 3: upsert

For each new row, ask one question: *is this order already in the table?*

- Yes: **update** that row.
- No: **insert** a new row.

Update plus insert: **upsert**. To answer the question, the database must know what makes two rows "the same order". That is the **key**. Here it is `order_id`.

Postgres can write an upsert as `INSERT ... ON CONFLICT`.

In [7]:
fresh_table("orders_upsert")
con.execute("ALTER TABLE orders_upsert ADD PRIMARY KEY (order_id)")      # the key

con.execute(f"""
    INSERT INTO orders_upsert {TUESDAY}
    ON CONFLICT (order_id) DO UPDATE
        SET status         = EXCLUDED.status,
            total_cents    = EXCLUDED.total_cents,
            refunded_cents = EXCLUDED.refunded_cents,
            updated_at     = EXCLUDED.updated_at
""")
describe("orders_upsert")

,table_name,rows,different_orders,rows_for_order_1130,refund_on_1130
0,orders_upsert,251,251,1,578


251 rows, 251 orders. One row for order 1130, and it carries the refund. `EXCLUDED` means "the incoming row".

## The same upsert, as MERGE

Most warehouses have one statement made for this: **`MERGE`**. Postgres has it from version 15. It reads almost like the two rules above.

In [8]:
fresh_table("orders_merge")
con.execute("ALTER TABLE orders_merge ADD PRIMARY KEY (order_id)")

con.execute(f"""
    MERGE INTO orders_merge AS t
    USING ({TUESDAY}) AS s
       ON t.order_id = s.order_id
    WHEN MATCHED THEN                                   -- the order is already here: update it
        UPDATE SET status = s.status, total_cents = s.total_cents,
                   refunded_cents = s.refunded_cents, updated_at = s.updated_at
    WHEN NOT MATCHED THEN                               -- a new order: insert it
        INSERT (order_id, status, order_date, total_cents, refunded_cents, updated_at)
        VALUES (s.order_id, s.status, s.order_date, s.total_cents, s.refunded_cents, s.updated_at)
""")
describe("orders_merge")

,table_name,rows,different_orders,rows_for_order_1130,refund_on_1130
0,orders_merge,251,251,1,578


The same result. `MATCHED` means the key exists. `NOT MATCHED` means it is new.

## Does the report agree with the truth now?

Add up each table by day, and hold it against a full read of the source.

In [9]:
import pandas as pd

truth = true_day_totals()

def report_from(name):
    rows = con.execute(f"""
        SELECT order_date::TEXT, count(*) FILTER (WHERE status <> 'cancelled'),
               coalesce(sum(total_cents - refunded_cents) FILTER (WHERE status <> 'cancelled'), 0)
        FROM {name} GROUP BY order_date
    """).fetchall()
    return {day: (orders, int(cents)) for day, orders, cents in rows}

result = []
for name in ["orders_append", "orders_overwrite", "orders_merge"]:
    report = report_from(name)
    wrong = [day for day, t in truth.items() if report.get(day, (0, 0)) != (t["orders"], t["revenue_cents"])]
    friday = report.get("2026-06-05", (0, 0))
    result.append({"written by": name.replace("orders_", ""), "Friday 5 June": f"{friday[0]} orders, {money(friday[1])}",
                   "days wrong": len(wrong), "": "✅" if not wrong else "❌"})

print("The truth for Friday 5 June:", truth["2026-06-05"]["orders"], "orders,", money(truth["2026-06-05"]["revenue_cents"]))
pd.DataFrame(result)

The truth for Friday 5 June: 37 orders, $971.04


,written by,Friday 5 June,days wrong,
0,append,"38 orders, $985.79",2,❌
1,overwrite,"1 orders, $8.97",7,❌
2,merge,"37 orders, $971.04",0,✅


Only the upsert gives the truth on every day.

In [10]:
show_doc("LLD.md", "4. The MERGE")

**From `docs/LLD.md`**

> ## 4. The MERGE
> 
> ```sql
> MERGE INTO core.orders AS t
> USING stage_orders AS s
>    ON t.order_id = s.order_id
> WHEN MATCHED AND s.updated_at > t.updated_at THEN      -- a newer version: update the row
>     UPDATE SET status = s.status, item_count = s.item_count,
>                total_cents = s.total_cents, refunded_cents = s.refunded_cents,
>                updated_at = s.updated_at, last_run_id = :run_id
> WHEN NOT MATCHED THEN                                   -- a new order: insert a row
>     INSERT (...) VALUES (...);
> ```
> 
> Three things can happen to an incoming row:
> 
> | The incoming order | Rule that fires | Result |
> |---|---|---|
> | Is not in the table | `NOT MATCHED` | Inserted |
> | Is in the table, and this version is newer | `MATCHED AND newer` | Updated |
> | Is in the table, with the same or an older version | none | **Left alone** |
> 
> The third row is what makes a second run harmless.
> 
> Order 1130 across two mornings:
> 
> | After run | Rows in `raw.orders_landing` | Row in `core.orders` |
> |---|---|---|
> | `2026-06-08_1` | 1 (refunded 0) | total 1475, refunded 0 |
> | `2026-06-09_1` | 2 (refunded 0, refunded 578) | total 1475, refunded **578** |
> | `2026-06-09_2` | 3 | total 1475, refunded 578: unchanged |

In [11]:
con.close()

## Two new words

| Word | Meaning | Here |
|---|---|---|
| **Upsert** | "An upsert means update plus insert. If the key already exists, the row is updated. If not, the row is added." | Tuesday's 29 rows: 4 updates and 25 inserts |
| **MERGE** | The SQL statement for an upsert. `MATCHED` means the key exists. `NOT MATCHED` means it is new | `MERGE INTO ... ON t.order_id = s.order_id` |

And two you met before, now side by side:

| Pattern | What it does | Fits |
|---|---|---|
| **Append** | Adds rows at the end | Raw data, where every version is kept |
| **Overwrite** | Empties the table and loads everything | A full load of a small table |
| **Upsert** | Updates a row if the key exists, adds it if not | A clean table, fed by an incremental load |

The definitions are from the D8Loop chapter [Loading & Write Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/loading-and-write-patterns).

## What you now know

- Append kept two rows for a changed order. Overwrite lost 222 orders. Upsert gave 251 rows for 251 orders.
- An upsert needs a key: the column that says two rows are the same thing.
- `MERGE` is the statement for an upsert. The report built on it equals the truth.

**Say it in an interview:** "My incremental load brought changed orders again, so an append double counted them. I switched the clean table to an upsert with MERGE on `order_id`: matched rows are updated, new rows are inserted. I kept append for the raw layer, because there I want every version."

Next: [notebook 2](02_which_column_is_the_key.ipynb) asks what happens when the key is wrong, or when the same key arrives twice in one run.